# Public Key Cryptography

Learn asymmetric encryption using RSA public/private key pairs.

In [2]:
from pathlib import Path

from Crypto.Cipher import PKCS1_OAEP
from Crypto.PublicKey import RSA

# Search for key files starting from current directory and moving up
def find_project_root():
    current = Path.cwd()
    # Try current directory first
    if (current / "my_keypair").exists():
        return current
    # Try parent directory
    if (current.parent / "my_keypair").exists():
        return current.parent
    # Try two levels up (for nested structures)
    if (current.parent.parent / "my_keypair").exists():
        return current.parent.parent
    return current  # Default to current if not found

PROJECT_FOLDER = find_project_root()
print(f"Looking for keypairs in folder: {PROJECT_FOLDER}")

PRIVATE_KEY_FILE = PROJECT_FOLDER / "my_keypair"
PUBLIC_KEY_FILE = PROJECT_FOLDER / "my_keypair.pub"

assert PRIVATE_KEY_FILE.exists(), f"Private key not found at {PRIVATE_KEY_FILE}. CWD: {Path.cwd()}"
assert PUBLIC_KEY_FILE.exists(), f"Public key not found at {PUBLIC_KEY_FILE}"

print(f"✓ Found private key: {PRIVATE_KEY_FILE}")
print(f"✓ Found public key: {PUBLIC_KEY_FILE}")


Looking for keypairs in folder: /workspaces/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services
✓ Found private key: /workspaces/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services/my_keypair
✓ Found public key: /workspaces/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services/my_keypair.pub


Load the private key from file
The private key must be kept secret and secure
Extract the public key from our private key
The public key can be freely shared with anyone


In [3]:
with open(PRIVATE_KEY_FILE, "r", encoding="utf8") as key_file:
    private_key = RSA.import_key(key_file.read())

public_key = private_key.publickey()
print(f"Public key:\n{public_key.export_key().decode('utf-8')}")


Public key:
-----BEGIN PUBLIC KEY-----
MIIBojANBgkqhkiG9w0BAQEFAAOCAY8AMIIBigKCAYEAi7cQZlqT3BujiEZd8EI4
jlWVFPCUSpilMvFH7csC2Slj+sUWoIzMGgkUI5NzTtaUJ9F6LjOugozi7DZNRhpD
5/wuo2wjFReRCaMlS6BfiiQuvR6BxCN1sGBAeNSuFtTFs4QFI2eRhzc0tZFj3T+1
ubT/DfuxS6l6e0tNkGqqHKk1rd5M8jJlGabijgTIaXEbV3LyzSveemqXdGQ7/tUb
1fR+M5IUSt07Ot93Sseusa9Rb3zOO9xa7fG+TIHA4bfn8u6pGGN3nLKwlUrp8b1n
sdXkFld6ul+ddCiTkjgM2dWLC9gbh6WE7DkK2RaNr+244kaqBvDJztCzsgisWlLY
vOxa6EpOrH5s+7xL8hRBJFWYXq8OsMmFr+TksiYUkTHg+x2tvrKz8IJpRrvIx1cM
4iIvcoy9SVkXAmRH+je26RK50MbSorgrLLMWOz55Y5owqe7jEjX2iC2D5+zG92v4
JdE+inj5CnYrQo6bMVc4bRo1WGa/hOZ9/KepLjL0yqLnAgMBAAE=
-----END PUBLIC KEY-----


Message to be encrypted - must be converted to bytes
Create a cipher object using the public key for encryption
Encrypt our message - only someone with the private key can decrypt it
Save the encrypted message to a file


In [4]:
short_secret_message = "My Secret Message".encode("utf-8")

public_key_cipher = PKCS1_OAEP.new(public_key)

encrypted_message = public_key_cipher.encrypt(short_secret_message)
print("Encrypted message:")
print(encrypted_message.hex())

ENCRYPTED_MESSAGE_FILE = PROJECT_FOLDER / "encrypted_message.bin"
with open(ENCRYPTED_MESSAGE_FILE, "wb") as f:
    f.write(encrypted_message)


Encrypted message:
7d34d36f4367410eb0540e2fdbae7396d70a25ff4c8bb6cb3c27d2f401272463b2679de7bf00ef17fbaa015d66f451d552bfa9f5bd3d30f41b958d0068f7a88ee4157a56f7cd1613789cfb8bdb7c20470956077eb727c6dc97795ed3fff1be39f22582ca2755ca70e60a8424b47f28c1687df1042307fd7af5f3fae510a5cb9484103ac0725c431fb71acb9ff6f201588f4b6e61f1f74e0903830183cbb9ea3ba280cb9c8b30c1ca7f9ef129978dbd39d121aca98013636d1f020db6aa4b626834fdbdcb3bd2144b52d84c725ed0e4f4e7a60e0a0cfd85eeedba8b8e25096846f0a29c9a4991bb9382b87110a69610775567c3fcb870c21f589ce86ff078b3ac7bd6a08d003f90eea9d16128919976f4d8d310bd6996591186c74ed567bd5e998688b6dfa56220c744c1fb774bc6c6375d9101b3bd5f9fcf1d8b54133116ad2b5e640654e94c71816ae69475ce1b6dbc2f0380deb34dd9cd6ecda38e3d5acfe14a5c799f76632e025130083193be157759227e3e0e38404352ff428d2ff548d1


Read the encrypted message from file
Create a cipher object using the private key for decryption
Decrypt the message using the private key
This demonstrates asymmetric encryption where:
1. Messages are encrypted with a public key and can only be decrypted with the matching private key
2. The public key can be freely shared
3. The private key must be kept secret
4. This is more secure than symmetric encryption as the decryption key (private key) is never shared


In [5]:
with open(ENCRYPTED_MESSAGE_FILE, "rb") as f:
    encrypted_message_from_file = f.read()

private_key_cipher = PKCS1_OAEP.new(private_key)

decrypted_message = private_key_cipher.decrypt(encrypted_message_from_file)
print(f"Decrypted message: {decrypted_message.decode('utf-8')}")



Decrypted message: My Secret Message
